# CestaData 3.0 — Experiência Prática IV
Banco SQLite isolado no Google Colab. Execute as células em ordem.

**Atenção:** a primeira célula apaga apenas `cestadata_3_0_ep4.db` do ambiente temporário do Colab para tornar os testes repetíveis. Faça backup do arquivo antes de reexecutá-la se tiver dados que deseja preservar.

In [ ]:
import sqlite3
from pathlib import Path
arquivo = Path('/content/cestadata_3_0_ep4.db')
if arquivo.exists():
    arquivo.unlink()  # reinicia SOMENTE o banco de testes deste notebook
conexao = sqlite3.connect(str(arquivo))
conexao.execute('PRAGMA foreign_keys = ON')
print('Banco de testes conectado:', arquivo.name)
print('Chaves estrangeiras ativas:', conexao.execute('PRAGMA foreign_keys').fetchone()[0])


## 1. Criar a estrutura (DDL)
Seis tabelas com PK, FK, NOT NULL, UNIQUE e CHECK.

In [ ]:
estrutura_sql = "-- CestaData 3.0 - Estrutura logica para SQLite\nPRAGMA foreign_keys = ON;\nCREATE TABLE IF NOT EXISTS Categoria (\n id_categoria INTEGER PRIMARY KEY,\n nome TEXT NOT NULL UNIQUE,\n descricao TEXT\n);\nCREATE TABLE IF NOT EXISTS Fornecedor (\n id_fornecedor INTEGER PRIMARY KEY,\n razao_social TEXT NOT NULL,\n cnpj TEXT UNIQUE,\n telefone TEXT,\n email TEXT\n);\nCREATE TABLE IF NOT EXISTS Usuario (\n id_usuario INTEGER PRIMARY KEY,\n nome TEXT NOT NULL,\n email TEXT UNIQUE,\n perfil TEXT NOT NULL CHECK (perfil IN ('Gestor','Estoquista','Cadastro','Administrador')),\n ativo INTEGER NOT NULL CHECK (ativo IN (0,1))\n);\nCREATE TABLE IF NOT EXISTS Produto (\n id_produto INTEGER PRIMARY KEY,\n nome TEXT NOT NULL,\n descricao TEXT,\n preco NUMERIC NOT NULL CHECK (preco >= 0),\n estoque_minimo INTEGER NOT NULL CHECK (estoque_minimo >= 0),\n id_categoria INTEGER NOT NULL,\n FOREIGN KEY (id_categoria) REFERENCES Categoria(id_categoria) ON DELETE RESTRICT\n);\nCREATE TABLE IF NOT EXISTS Produto_Fornecedor (\n id_produto INTEGER NOT NULL,\n id_fornecedor INTEGER NOT NULL,\n PRIMARY KEY (id_produto,id_fornecedor),\n FOREIGN KEY (id_produto) REFERENCES Produto(id_produto) ON DELETE RESTRICT,\n FOREIGN KEY (id_fornecedor) REFERENCES Fornecedor(id_fornecedor) ON DELETE RESTRICT\n);\nCREATE TABLE IF NOT EXISTS Movimentacao_Estoque (\n id_movimentacao INTEGER PRIMARY KEY,\n id_produto INTEGER NOT NULL,\n id_usuario INTEGER NOT NULL,\n tipo TEXT NOT NULL CHECK (tipo IN ('ENTRADA','SAIDA')),\n quantidade INTEGER NOT NULL CHECK (quantidade > 0),\n data_movimentacao TEXT NOT NULL,\n FOREIGN KEY (id_produto) REFERENCES Produto(id_produto) ON DELETE RESTRICT,\n FOREIGN KEY (id_usuario) REFERENCES Usuario(id_usuario) ON DELETE RESTRICT\n);"
conexao.executescript(estrutura_sql)
print('Seis tabelas criadas com sucesso.')


## 2. Popular o banco (DML — INSERT)
Uma transação confirma os 50 registros. Em caso de erro, a conexão é refeita com banco novo usando a primeira célula.

In [ ]:
povoamento_sql = "-- CestaData 3.0 - Povoamento inicial: executar apenas uma vez em banco vazio.\n-- As entidades independentes sao inseridas antes das dependentes.\nBEGIN TRANSACTION;\n-- 1. Categorias\nINSERT INTO Categoria (id_categoria,nome,descricao) VALUES\n(1,'Alimentos','Produtos alimenticios basicos'),\n(2,'Bebidas','Produtos para consumo liquido'),\n(3,'Mercearia','Produtos secos e complementares'),\n(4,'Laticinios','Leites e derivados');\n-- 2. Fornecedores (identificadores ficticios; nao sao CNPJs validos)\nINSERT INTO Fornecedor (id_fornecedor,razao_social,cnpj,telefone,email) VALUES\n(1,'Distribuidora Alfa','00000000000001','11999990001','alfa@exemplo.com'),\n(2,'Atacado Beta','00000000000002','11999990002','beta@exemplo.com'),\n(3,'Comercial Gama','00000000000003','11999990003','gama@exemplo.com'),\n(4,'Distribuidora Delta','00000000000004','11999990004','delta@exemplo.com'),\n(5,'Atacado Omega','00000000000005','11999990005','omega@exemplo.com');\n-- 3. Usuarios\nINSERT INTO Usuario (id_usuario,nome,email,perfil,ativo) VALUES\n(1,'Ana Silva','ana@exemplo.com','Gestor',1),\n(2,'Carlos Souza','carlos@exemplo.com','Estoquista',1),\n(3,'Mariana Lima','mariana@exemplo.com','Cadastro',1),\n(4,'Pedro Santos','pedro@exemplo.com','Administrador',1);\n-- 4. Produtos: mesmos precos do CestaData 2.0\nINSERT INTO Produto (id_produto,nome,descricao,preco,estoque_minimo,id_categoria) VALUES\n(1,'Arroz','Pacote de arroz 5kg',24.90,10,1),\n(2,'Feijao','Pacote de feijao 1kg',8.50,10,1),\n(3,'Cafe','Pacote de cafe 500g',19.90,8,3),\n(4,'Acucar','Pacote de acucar 1kg',5.20,10,3),\n(5,'Oleo','Oleo vegetal 900ml',7.80,10,3),\n(6,'Leite','Leite integral 1L',5.90,10,4),\n(7,'Macarrao','Macarrao 500g',4.50,12,1),\n(8,'Farinha','Farinha de trigo 1kg',6.70,10,3),\n(9,'Sal','Pacote de sal 1kg',3.20,8,3),\n(10,'Margarina','Margarina 500g',9.90,10,4);\n-- 5. Associacoes N:M\nINSERT INTO Produto_Fornecedor (id_produto,id_fornecedor) VALUES\n(1,1),(1,2),(2,1),(2,3),(3,2),(4,3),(5,4),(6,5),(7,2),(8,4),(9,3),(10,5);\n-- 6. Movimentacoes: entradas iniciais, depois saidas\nINSERT INTO Movimentacao_Estoque (id_movimentacao,id_produto,id_usuario,tipo,quantidade,data_movimentacao) VALUES\n(1,1,2,'ENTRADA',50,'2026-11-01 09:00:00'),\n(2,2,2,'ENTRADA',40,'2026-11-01 09:10:00'),\n(3,3,2,'ENTRADA',30,'2026-11-01 09:20:00'),\n(4,4,2,'ENTRADA',45,'2026-11-01 09:30:00'),\n(5,5,2,'ENTRADA',35,'2026-11-01 09:40:00'),\n(6,6,2,'ENTRADA',60,'2026-11-01 09:50:00'),\n(7,7,2,'ENTRADA',50,'2026-11-01 10:00:00'),\n(8,8,2,'ENTRADA',25,'2026-11-01 10:10:00'),\n(9,9,2,'ENTRADA',30,'2026-11-01 10:20:00'),\n(10,10,2,'ENTRADA',20,'2026-11-01 10:30:00'),\n(11,1,2,'SAIDA',12,'2026-11-02 14:00:00'),\n(12,2,2,'SAIDA',8,'2026-11-02 14:10:00'),\n(13,5,2,'SAIDA',6,'2026-11-02 14:20:00'),\n(14,6,2,'SAIDA',15,'2026-11-02 14:30:00'),\n(15,10,2,'SAIDA',5,'2026-11-02 14:40:00');\nCOMMIT;"
conexao.executescript(povoamento_sql)
print('Carga de dados concluida.')


## 3. Conferir quantidade de registros e relacionamentos

In [ ]:
tabelas = ['Categoria','Fornecedor','Usuario','Produto','Produto_Fornecedor','Movimentacao_Estoque']
for tabela in tabelas:
    qtd = conexao.execute(f'SELECT COUNT(*) FROM {tabela}').fetchone()[0]
    print(f'{tabela}: {qtd} registro(s)')
print('Integridade referencial:', 'OK' if not conexao.execute('PRAGMA foreign_key_check').fetchall() else 'ERRO')


## 4. Consulta com JOIN, WHERE, ORDER BY e LIMIT

In [ ]:
consulta = """SELECT p.nome AS produto, c.nome AS categoria, p.preco
FROM Produto AS p
JOIN Categoria AS c ON c.id_categoria = p.id_categoria
WHERE p.preco > 10
ORDER BY p.preco DESC
LIMIT 5"""
for produto, categoria, preco in conexao.execute(consulta):
    print(f'{produto:12s} | {categoria:12s} | R$ {preco:.2f}')


## 5. Consultar os saldos calculados pelas movimentações

In [ ]:
sql_saldos = """SELECT p.nome, p.estoque_minimo,
COALESCE(SUM(CASE WHEN m.tipo='ENTRADA' THEN m.quantidade
                  WHEN m.tipo='SAIDA' THEN -m.quantidade END),0) AS saldo
FROM Produto p LEFT JOIN Movimentacao_Estoque m ON p.id_produto=m.id_produto
GROUP BY p.id_produto,p.nome,p.estoque_minimo
ORDER BY p.id_produto"""
for nome, minimo, saldo in conexao.execute(sql_saldos):
    print(f'{nome:12s} | Saldo: {saldo:3d} | Minimo: {minimo:2d}')


## 6. Teste seguro de integridade referencial
Tentativa de inserir produto com categoria inexistente. A falha é esperada; nenhum registro inválido será mantido.

In [ ]:
try:
    conexao.execute("""INSERT INTO Produto
    (id_produto,nome,preco,estoque_minimo,id_categoria)
    VALUES (99,'Teste invalido',5.00,1,999)""")
    conexao.rollback()
    print('ATENCAO: a insercao foi aceita; verifique as FKs')
except sqlite3.IntegrityError as erro:
    conexao.rollback()
    print('Teste aprovado: insercao bloqueada pelo banco.')
    print('Motivo:', erro)


**Nota técnica:** `COMMIT` confirma uma transação SQL; não é um commit do Git. No SQLite, `NUMERIC` não garante precisão monetária exata. Além disso, impedir saídas que excedam o saldo requer uma validação adicional que não é garantida apenas pelas FKs.